# 3A Een large language model gebouwd in NumPy

Hieronder wordt een klein taalmodel gebaseerd op de Transformer-architectuur (https://proceedings.neurips.cc/paper/2017/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html) - die alle beroemde LLM's zoals ChatGPT, Gemini, DeepSeek, etc. gebruiken - uitgewerkt, volledig in NumPy. Bekijk ook zeker de slides om theoretisch te snappen wat hier en daar berekend wordt en waarom.

We importeren eerst NumPy:

In [1]:
import numpy as np

We maken een functie die de zinnen omzet in een nummer-encodering. Elk woord krijgt een index, opgeslagen in dictionary `stoi` (string-to-integer) en een omgekeerde versie `itos` (integer-to-string) om later de voorspelde nummers terug om te zetten in woorden.

In [2]:
def bouw_vocab(zinnen):
    woorden = set()
    for z in zinnen:
        woorden.update(z.split())
    woorden = sorted(woorden)
    stoi = {w: i for i, w in enumerate(woorden)}
    itos = {i: w for w, i in stoi.items()}
    return stoi, itos

Voorbeeld:

In [3]:
zinnen = [
    "ik eet pizza",
    "ik hou van de alma",
    "AI houdt niet van de alma",
    "ik eet soms pizza en pasta",
    "AI eet geen pizza"
]

stoi, itos = bouw_vocab(zinnen)
print(stoi)
print(itos)

{'AI': 0, 'alma': 1, 'de': 2, 'eet': 3, 'en': 4, 'geen': 5, 'hou': 6, 'houdt': 7, 'ik': 8, 'niet': 9, 'pasta': 10, 'pizza': 11, 'soms': 12, 'van': 13}
{0: 'AI', 1: 'alma', 2: 'de', 3: 'eet', 4: 'en', 5: 'geen', 6: 'hou', 7: 'houdt', 8: 'ik', 9: 'niet', 10: 'pasta', 11: 'pizza', 12: 'soms', 13: 'van'}


We voegen informatie over de positie van een token in een zin toe met de volgende functie. Dit is een eenvoudige encodering gebaseerd op de sinus van een reeks. In grote modellen wordt deze mee geleerd (ook tussen zinnen). Dit maakt dat het model wel effectief een verschil zal kunnen leren tussen 'ik eet' en 'eet ik'.

In [4]:
def positionele_encodering(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    delers = np.power(10000, (2 * (i//2)) / d_model)
    hoeken = pos / delers
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(hoeken[:, 0::2])
    pe[:, 1::2] = np.cos(hoeken[:, 1::2])
    return pe

Voorbeeld:

In [5]:
positionele_encodering(5, 4)

array([[ 0.        ,  1.        ,  0.        ,  1.        ],
       [ 0.84147098,  0.54030231,  0.00999983,  0.99995   ],
       [ 0.90929743, -0.41614684,  0.01999867,  0.99980001],
       [ 0.14112001, -0.9899925 ,  0.0299955 ,  0.99955003],
       [-0.7568025 , -0.65364362,  0.03998933,  0.99920011]])

We maken ook gebruik van de softmax functie $$softmax(x) = \frac{e^{x_i}}{\Sigma_{j=1}^K e^{x_i}}$$ die een numerieke vector $x$ omzet in een vector van probabiliteiten. Deze functie wordt gebruikt om op het einde van het model een voorspelling per woord in het gehele vocabularium te maken zodat we het woord met de hoogste kans om het volgende woord te zijn kunnen selecteren. Het wordt ook gebruikt om de attentie-scores te berekenen. Hier implementeren we een numeriek stabiele versie met een maximum zodat de waarschijnlijkheden niet 'ontploffen' door grote getallen.

In [6]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

Voorbeeld:

In [7]:
softmax([10, 20, 30, 50])

array([4.24835425e-18, 9.35762295e-14, 2.06115362e-09, 9.99999998e-01])

Ooit al gehoord van de `temperature` van een LLM waarmee je 'creatievere' antwoorden kan krijgen? Dit gebeurt door de softmax-functie aan te passen met een extra parameter `T` die de gegenereerde kansen beter verdeeld over de mogelijke uitkomsttokens en zo ervoor zorgt dat minder frequente tokens ook aan bod kunnen komen. Als `T>1` zal de verdeling dus meer uniform worden en waarden zoals `T=100` zal zelfs een quasi uniforme verdeling benaderen.

In [8]:
T = 10

def softmax_met_temperatuur(x):
    x = (np.array(x) / T)
    e = np.exp(x - np.max(x))
    return (e) / e.sum()

softmax_met_temperatuur([10, 20, 30, 50])

array([0.01521943, 0.0413707 , 0.11245721, 0.83095266])

Nu brengen we alles samen in een Transformer-klasse. Ons model heeft het token-vocabularium, sequentie-lengte waarop die getraind is (zie ook slides voor uitleg omtrent het model), een dimensionaliteit (hoe groter, hoe meer er kan 'geleerd' worden) en een learning rate (hoe snel het model probeert te convergeren - te hoog: neigt te snel naar lokaal optimum, te laag: neigt nooit naar optimum). 

Daarnaast worden de parameters ook geïnitialiseerd. Dit zijn de gewichten in het neurale netwerk die de input aan de output linken en die we zullen 'leren' om ons model zo goed mogelijk de tekst te laten genereren door telkens het volgende woord te voorspellen.

In [ ]:
class MiniTransformer:
    def __init__(self, vocab_grootte, seq_len, d_model=16, lr=0.05, seed=0):
        np.random.seed(seed)
        self.vocab = vocab_grootte
        self.seq_len = seq_len
        self.d_model = d_model
        self.lr = lr

        # te leren parameters worden willekeurig gezet
        # E zijn de embeddings, die de semantische representatie van woorden leren
        self.E = np.random.randn(vocab_grootte, d_model) * 0.1
        # dit zijn de gewichten voor query, key en value
        self.WQ = np.random.randn(d_model, d_model) * 0.1
        self.WK = np.random.randn(d_model, d_model) * 0.1
        self.WV = np.random.randn(d_model, d_model) * 0.1
        # deze matrix zet de geleerde 'middensectie' om naar kansen per
        # element in het vocabularium
        self.WO = np.random.randn(d_model, vocab_grootte) * 0.1

        self.PE = positionele_encodering(seq_len, d_model)

    def attention(self, X):
        # X is een matrix van embeddings met shape (seq_len, d_model)
        # en stelt dus een sequentie van woorden voor
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        scores = (Q @ K.T) / np.sqrt(self.d_model)
        mask = np.tril(np.ones_like(scores))
        scores = np.where(mask == 0, -1e9, scores)

        w = softmax(scores)
        return w @ V, w

    # 1 iteratie over de data waarin alle gewichten worden gebruikt om een voorspelling te maken
    def forward(self, seq_idx):
        # we halen de embeddings op en voegen de positionele encodering toe
        X = self.E[seq_idx] + self.PE[:len(seq_idx)]
        # we passen de attentie toe op alle embeddings in de sequentie
        H, att = self.attention(X)
        logits = H @ self.WO
        return logits[-1], H, att

    # leren van de gewichten over de data
    def train_step(self, seq, target):
        # we doen een 'pass' over de sequenties om de kansen te voorspellen
        logits, H, att = self.forward(seq)
        probs = softmax(logits)

        # afgeleiden opslaan en target index resetten
        dlogits = probs.copy()
        dlogits[target] -= 1  

        # we berekenen de afgeleiden die de gewichten zo goed mogelijk 'richten' 
        # om een betere benadering te maken van onze output vanuit ons model
        # hiervoor gebruik we het 'buiten'product (<> binnenproduct is dot)
        dWO = np.outer(H[-1], dlogits)

        # we passen onze gewichten aan na deze trainingsronde
        self.WO -= self.lr * dWO

        # we berekenen hoeveel ons model er nog 'naastzit'
        verlies = -np.log(probs[target] + 1e-9)
        return verlies


Nu gaan we onze data opzetten om ons model te trainen. We maken verschillende trainings-paren aan zodat we elk woord in verschillende contexten tegenkomen (zie ook de introductieslides). Zo leert het model het woord te voorspellen na verschillende andere woorden en woordencombinaties. Merk op dat de woorden hier al zijn omgezet in hun index uit `stoi`. Hoe meer data je zou geven, hoe meer contexten het model kan leren.

In [10]:
zinnen = [
    "ik eet pizza",
    "ik hou van de alma",
    "AI houdt niet van de alma",
    "ik eet soms pizza en pasta",
    "AI eet geen pizza"
]

# bouwen van vocabularium
stoi, itos = bouw_vocab(zinnen)

# model aanmaken
model = MiniTransformer(vocab_grootte=len(stoi), seq_len=5, d_model=16, lr=0.1)

# trainingdata maken (telkens de zinnen zo opbouwen dat we het volgende woord leren voorspellen
train_pairs = []
for zin in zinnen:
    woorden = zin.split()
    for i in range(1, len(woorden)):
        context = woorden[:i]
        target = woorden[i]
        seq = [stoi[w] for w in context]
        train_pairs.append((seq, stoi[target]))
print(train_pairs)


[([8], 3), ([8, 3], 11), ([8], 6), ([8, 6], 13), ([8, 6, 13], 2), ([8, 6, 13, 2], 1), ([0], 7), ([0, 7], 9), ([0, 7, 9], 13), ([0, 7, 9, 13], 2), ([0, 7, 9, 13, 2], 1), ([8], 3), ([8, 3], 12), ([8, 3, 12], 11), ([8, 3, 12, 11], 4), ([8, 3, 12, 11, 4], 10), ([0], 3), ([0, 3], 5), ([0, 3, 5], 11)]


Nu gaan we het model de data aanbrengen en iteratief de gewichten in het model aanpassen zodat we, geleid door het `verlies` dat het verschil uitdrukt tussen het voorspelde woord van het model en het eigenlijke woord uit de data, gaandeweg de trainingdata beter en beter gaan benaderen.

In [11]:
for epoch in range(50):
    verliezen = []
    np.random.shuffle(train_pairs)
    for seq, tgt in train_pairs:
        verlies = model.train_step(seq, tgt)
        verliezen.append(verlies)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Gemiddeld verlies: {np.mean(verliezen):.4f}")

Epoch   0 | Gemiddeld verlies: 2.6335
Epoch  10 | Gemiddeld verlies: 2.4224
Epoch  20 | Gemiddeld verlies: 2.3718
Epoch  30 | Gemiddeld verlies: 2.3416
Epoch  40 | Gemiddeld verlies: 2.3179


Het verlies zakt mooi. Probeer zelf eens wat meer epochs uit om te zien of het verlies nog meer naar beneden kan.

Nu gaan we ons model eens een aantal voorspellingen laten doen:

In [12]:
context = "ik hou van"
tokens = context.split()
seq = [stoi[w] for w in tokens]

# laat ons model de sequentie 'inlezen'
logits, _, _ = model.forward(seq)
# het model kan nu een voorspelling maken voor het
# meest waarschijnlijke volgende woord
probs = softmax(logits)

# we kijken naar de hoogste waarschijnlijkheden:
print("\nContext:", context)
print("Voorspelling:", itos[np.argmax(probs)])
print("\nTop 3:")
for idx in probs.argsort()[-3:][::-1]:
    print(f"  {itos[idx]:12s} p={probs[idx]:.3f}")


Context: ik hou van
Voorspelling: eet

Top 3:
  eet          p=0.140
  pizza        p=0.132
  van          p=0.101


Ziezo, je hebt je eigen ChatGPT gemaakt. Voeg nu terrabytes aan tekstuele data toe en een parallelliseerbare architectuur die over 20,000 GPU's wordt verdeeld en je kan beginnen met je eigen startup.

Merk vooral op hoe NumPy ons hier in vele aspecten helpt door de vectoren aan te maken, de matrix-vermenigvuldiging te ondersteunen en alle mathematische functies als pakket aan te bieden (en te optimaliseren in de achtergrond).

Laten we dit eens op het script van Lord of the Rings - The Fellowship of the Ring proberen. Het script zit in het bijgevoegde tekstbestand `LORT1.txt` en is te vinden op bv. https://imsdb.com/scripts/Lord-of-the-Rings-Fellowship-of-the-Ring,-The.html.

Deze tekst in uiteraard heel wat groter en complexer:

In [13]:
bestand = open('LOTR1.txt', 'r')
text = bestand.read()
bestand.close()

zinnen = text.split('\n')
print('Aantal zinnen in tekst:', len(zinnen))
zinnen = [z for z in zinnen if len(z) > 0]
print('Aantal zinnen na filteren:', len(zinnen))
print('Aantal woorden in vocabularium:', len(bouw_vocab(zinnen)[0]))
print('Langste zin:', max(len(z.split()) for z in zinnen))
zinnen

Aantal zinnen in tekst: 6949
Aantal zinnen na filteren: 4819
Aantal woorden in vocabularium: 6836
Langste zin: 14


['BLACK SCREEN',
 'SUPER: New Line Cinema Presents',
 'SUPER: A Wingnut Films Production',
 "BLACK CONTINUES... ELVISH SINGING....A WOMAN'S VOICE IS",
 'whispering, tinged with SADNESS and REGRET:',
 '                    GALADRIEL (V.O.)',
 '              (Elvish: subtitled)',
 '          "I amar prestar sen: han mathon ne nen,',
 '          han mathon ne chae...a han noston ned',
 '          wilith."',
 '              (English:)',
 '          The world is changed: I feel it in the',
 '          water, I feel it in the earth, I smell it',
 '          in the air...Much that once was is lost,',
 '          for none now live who remember it.',
 'SUPER: THE LORD OF THE RINGS',
 'EXT. PROLOGUE -- DAY',
 'IMAGE: FLICKERING FIRELIGHT. The NOLDORIN FORGE in EREGION.',
 'MOLTEN GOLD POURS from the lip of an IRON LADLE.',
 '                    GALADRIEL (V.O.)',
 '          It began with the forging of the Great',
 '          Rings.',
 'IMAGE: THREE RINGS, each set with a single GEM, are receive

We bouwen hetzelfde model, maar nu met een langere sequentie-lengte en een grotere dimensie zodat we meer informatie kunnen capteren uit de tekst:

In [14]:
stoi, itos = bouw_vocab(zinnen)
model = MiniTransformer(vocab_grootte=len(stoi), seq_len=14, d_model=64, lr=0.1)

train_pairs = []
for zin in zinnen:
    woorden = zin.split()
    for i in range(1, len(woorden)):
        context = woorden[:i]
        target = woorden[i]
        seq = [stoi[w] for w in context]
        train_pairs.append((seq, stoi[target]))

print('Aantal trainingsparen:', len(train_pairs))


Aantal trainingsparen: 19613


We trainen het model (pas op, dit duurde 50min):

In [15]:
for epoch in range(50):
    verliezen = []
    np.random.shuffle(train_pairs)
    for seq, tgt in train_pairs:
        verlies = model.train_step(seq, tgt)
        verliezen.append(verlies)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Gemiddeld verlies: {np.mean(verliezen):.4f}")

Epoch   0 | Gemiddeld verlies: 7.5864
Epoch  10 | Gemiddeld verlies: 6.9918
Epoch  20 | Gemiddeld verlies: 6.8507
Epoch  30 | Gemiddeld verlies: 6.7546
Epoch  40 | Gemiddeld verlies: 6.6688


Nu kunnen we het model tekst laten genereren:

In [16]:
context = "Gandalf is"
tokens = context.split()
seq = [stoi[w] for w in tokens]
logits, _, _ = model.forward(seq)
probs = softmax(logits)

# we kijken naar de hoogste waarschijnlijkheden:
print("\nContext:", context)
print("Voorspelling:", itos[np.argmax(probs)])
print("\nTop 3:")
for idx in probs.argsort()[-3:][::-1]:
    print(f"  {itos[idx]:12s} p={probs[idx]:.3f}")


Context: Gandalf is
Voorspelling: a

Top 3:
  a            p=0.060
  the          p=0.057
  and          p=0.032


Nu laat ik het model iteratief de 10 volgende woorden genereren (en ja, het model zin meteen vast wegens te weinig trainingsdata):

In [17]:
context = "Frodo and Sam are"

# 10 woorden voorspellen
for i in range(10):
    tokens = context.split()
    seq = [stoi[w] for w in tokens]
    logits, _, _ = model.forward(seq)
    probs = softmax(logits)

    # we kijken naar de hoogste waarschijnlijkheden:
    print("\nContext:", context)
    print("Voorspelling:", itos[np.argmax(probs)])
    print("\nTop 3:")
    for idx in probs.argsort()[-3:][::-1]:
        print(f"  {itos[idx]:12s} p={probs[idx]:.3f}")
    context = context + ' ' + itos[np.argmax(probs)]


Context: Frodo and Sam are
Voorspelling: the

Top 3:
  the          p=0.042
  a            p=0.025
  to           p=0.023

Context: Frodo and Sam are the
Voorspelling: the

Top 3:
  the          p=0.040
  of           p=0.023
  and          p=0.021

Context: Frodo and Sam are the the
Voorspelling: the

Top 3:
  the          p=0.053
  of           p=0.032
  and          p=0.026

Context: Frodo and Sam are the the the
Voorspelling: the

Top 3:
  the          p=0.055
  of           p=0.032
  and          p=0.025

Context: Frodo and Sam are the the the the
Voorspelling: the

Top 3:
  the          p=0.036
  of           p=0.021
  and          p=0.017

Context: Frodo and Sam are the the the the the
Voorspelling: the

Top 3:
  the          p=0.018
  of           p=0.011
  to           p=0.009

Context: Frodo and Sam are the the the the the the
Voorspelling: the

Top 3:
  the          p=0.010
  of           p=0.006
  to           p=0.006

Context: Frodo and Sam are the the the the the the the